# Module 1: Preflight Checks

## Overview

This notebook validates your environment before deploying LangSmith. Most self-hosted failures occur **before** users ever touch the product due to:

- Mis-sized clusters
- Unsupported ingress setups
- In-cluster databases used past their limits
- Missing storage primitives (blob, PVs)

This preflight ensures you start from a **supported baseline**.

## What We'll Check

1. ✅ Tooling validation (cloud CLI, terraform, kubectl, helm, jq)
2. ✅ Cloud provider credentials & region sanity check
3. ✅ Cluster capacity expectations
4. ✅ Storage prerequisites (CSI drivers, StorageClasses)
5. ✅ Blob storage requirement (cloud object storage)

**Estimated time:** 20-30 minutes

**Supported Cloud Providers:** AWS, Azure (GCP coming soon)


In [ ]:
# Bootstrap environment
import sys
from pathlib import Path

# Add notebooks directory to path so we can import shared as a package
# Find the notebooks directory by looking for the shared folder
possible_paths = [
    Path.cwd().parent,  # If cwd is module-1, go up one level to notebooks
    Path.cwd(),  # If cwd is already notebooks
    Path.cwd() / "notebooks",  # If cwd is workspace root
]

notebooks_path = None
for path in possible_paths:
    if path and (path / "shared" / "_bootstrap.py").exists():
        notebooks_path = path
        break

if not notebooks_path:
    # Fallback: try workspace root
    notebooks_path = Path.cwd() / "notebooks"
    if not (notebooks_path / "shared" / "_bootstrap.py").exists():
        raise RuntimeError(f"Could not find notebooks/shared directory. Current dir: {Path.cwd()}")

# Add notebooks directory to path so 'shared' can be imported as a package
if str(notebooks_path) not in sys.path:
    sys.path.insert(0, str(notebooks_path))

from shared._bootstrap import bootstrap

# Run bootstrap: loads env, checks tools, validates AWS, creates artifacts dir
bootstrap_info = bootstrap()
print(f"\nBootstrap complete! Artifacts directory: {bootstrap_info['artifacts_dir']}")


## Cloud Provider Account & Region Validation

Verify you're using the correct cloud provider account/subscription and region. This is critical for avoiding accidental deployments to production or wrong regions.


In [ ]:
import os
import json
from shared._cloud_helpers import (
    get_cloud_provider,
    get_region,
    get_identity,
    assert_account,
)
from shared._validation import require_env, print_config, ok, warn

# Get cloud configuration
provider = get_cloud_provider()
region = get_region()
identity = get_identity()

provider_display = provider.upper()
print(f"### Current {provider_display} Session")
print(f"Cloud Provider: {provider_display}")
print(f"Region: {region}")

if provider == "aws":
    print(f"Account ID: {identity['Account']}")
    print(f"User ARN: {identity['Arn']}")
    account_var = "AWS_ACCOUNT_ID"
elif provider == "azure":
    subscription_id = identity.get("SubscriptionId") or identity.get("Account", "")
    subscription_name = identity.get("SubscriptionName", "")
    print(f"Subscription ID: {subscription_id}")
    print(f"Subscription Name: {subscription_name}")
    account_var = "AZURE_SUBSCRIPTION_ID"
else:
    account_var = None

# Optional: Validate against expected account/subscription
if account_var:
    expected_account = os.environ.get(account_var, "").strip()
    if expected_account:
        assert_account(expected_account)
    else:
        warn(f"{account_var} not set in environment - skipping account validation")
        print(f"💡 Tip: Set {account_var} in your .env file to add a guardrail against wrong account deployments")


## Required Environment Variables

Verify that all required configuration is present. These values will be used throughout the deployment.


In [ ]:
# Check required environment variables
from shared._cloud_helpers import get_cloud_provider

provider = get_cloud_provider()

# Base required vars (cloud-agnostic)
required_vars = [
    "WORKSHOP_NAME",
    "NAMESPACE",
    "CLUSTER_NAME",
    "TERRAFORM_DIR",
    "HELM_RELEASE",
    "HELM_NAMESPACE",
    "HELM_CHART_REF",
]

# Add cloud-specific required vars
if provider == "aws":
    required_vars.append("AWS_REGION")
elif provider == "azure":
    required_vars.append("AZURE_LOCATION")

config = require_env(*required_vars)

# Optional but recommended (cloud-specific)
optional_vars = {}
if provider == "aws":
    optional_vars = {
        "AWS_PROFILE": os.environ.get("AWS_PROFILE", ""),
        "AWS_ACCOUNT_ID": os.environ.get("AWS_ACCOUNT_ID", ""),
        "VALUES_FILE": os.environ.get("VALUES_FILE", ""),
    }
elif provider == "azure":
    optional_vars = {
        "AZURE_SUBSCRIPTION_ID": os.environ.get("AZURE_SUBSCRIPTION_ID", ""),
        "AZURE_RESOURCE_GROUP": os.environ.get("AZURE_RESOURCE_GROUP", ""),
        "VALUES_FILE": os.environ.get("VALUES_FILE", ""),
    }

print("\n### Configuration Summary")
print(f"Cloud Provider: {provider.upper()}")
print_config(config, redact_keys={"AWS_PROFILE"})
print("\n### Optional Configuration")
for k, v in optional_vars.items():
    if v:
        print(f"- {k}: {v}")
    else:
        print(f"- {k}: (not set)")


## Cluster Capacity Expectations

LangSmith requires adequate cluster resources. Before deploying, understand what you'll need:

- **Minimum:** 3 nodes, 4 vCPU, 16GB RAM each (for development/testing)
- **Recommended:** 3 nodes, 8 vCPU, 32GB RAM each (for production workloads)
- **Storage:** EBS CSI driver required for ClickHouse PVCs

Let's check if a cluster already exists and validate its configuration.


In [ ]:
from shared._cloud_helpers import (
    get_cloud_provider,
    get_region,
    cluster_exists,
    get_kubernetes_service_name,
)
from shared._shell import run

provider = get_cloud_provider()
cluster_name = os.environ["CLUSTER_NAME"]
region = get_region()
k8s_service = get_kubernetes_service_name()

print(f"### Checking {k8s_service} Cluster: {cluster_name}")
print(f"Cloud Provider: {provider.upper()}")
print(f"Region: {region}\n")

if cluster_exists(cluster_name):
    ok(f"Cluster '{cluster_name}' exists")
    
    # Get cluster details (cloud-specific)
    if provider == "aws":
        result = run(
            ["aws", "eks", "describe-cluster", "--name", cluster_name, "--region", region, "--output", "json"],
            check=True,
            stream=False
        )
        cluster_info = json.loads(result.stdout)["cluster"]
        
        print(f"\nCluster Status: {cluster_info.get('status', 'N/A')}")
        print(f"Kubernetes Version: {cluster_info.get('version', 'N/A')}")
        print(f"Platform Version: {cluster_info.get('platformVersion', 'N/A')}")
        
        # Check node groups (AWS-specific)
        print("\n### Node Groups")
        ng_result = run(
            ["aws", "eks", "list-nodegroups", "--cluster-name", cluster_name, "--region", region, "--output", "json"],
            check=True,
            stream=False
        )
        nodegroups = json.loads(ng_result.stdout).get("nodegroups", [])
        
        if nodegroups:
            for ng in nodegroups:
                ng_detail = run(
                    ["aws", "eks", "describe-nodegroup", "--cluster-name", cluster_name, 
                     "--nodegroup-name", ng, "--region", region, "--output", "json"],
                    check=True,
                    stream=False
                )
                ng_info = json.loads(ng_detail.stdout)["nodegroup"]
                scaling = ng_info.get("scalingConfig", {})
                print(f"\n  Node Group: {ng}")
                print(f"    Status: {ng_info['status']}")
                print(f"    Desired: {scaling.get('desiredSize', 'N/A')}")
                print(f"    Min: {scaling.get('minSize', 'N/A')}")
                print(f"    Max: {scaling.get('maxSize', 'N/A')}")
                print(f"    Instance Types: {', '.join(ng_info.get('instanceTypes', []))}")
        else:
            warn("No node groups found")
            print("💡 You'll need to create node groups when deploying with Terraform")
            
    elif provider == "azure":
        resource_group = os.environ.get("AZURE_RESOURCE_GROUP", cluster_name)
        result = run(
            ["az", "aks", "show", "--name", cluster_name, "--resource-group", resource_group, "--output", "json"],
            check=True,
            stream=False
        )
        cluster_info = json.loads(result.stdout)
        
        print(f"\nCluster Status: {cluster_info.get('powerState', {}).get('code', 'N/A')}")
        print(f"Kubernetes Version: {cluster_info.get('kubernetesVersion', 'N/A')}")
        print(f"Provisioning State: {cluster_info.get('provisioningState', 'N/A')}")
        print(f"Node Resource Group: {cluster_info.get('nodeResourceGroup', 'N/A')}")
        print("\n💡 For detailed node pool information, check the Azure portal or use: az aks nodepool list")
else:
    warn(f"Cluster '{cluster_name}' does not exist yet")
    print("💡 This is expected if you haven't run Terraform yet. Proceed to notebook 02_terraform_apply.ipynb")


## Storage Prerequisites

LangSmith requires persistent storage for ClickHouse. The cloud storage CSI driver must be installed and StorageClasses must be configured.

**Why this matters:** Without the appropriate CSI driver, ClickHouse PVCs will remain in `Pending` state forever.


In [ ]:
# Check if kubectl is configured for the cluster
from shared._cloud_helpers import (
    get_cloud_provider,
    get_region,
    configure_kubectl,
    get_storage_driver_name,
)

provider = get_cloud_provider()
cluster_name = os.environ["CLUSTER_NAME"]
region = get_region()
storage_driver = get_storage_driver_name()

k8s_service = "EKS" if provider == "aws" else "AKS" if provider == "azure" else "Kubernetes"
print(f"### Configuring kubectl for {k8s_service} cluster")
try:
    # Configure kubectl (cloud-agnostic)
    configure_kubectl(cluster_name, region)
    ok("kubectl configured for cluster")
    
    # Check CSI driver (cloud-specific labels)
    print(f"\n### Checking {storage_driver} Driver")
    
    if provider == "aws":
        driver_label = "app=ebs-csi-controller"
        driver_name = "EBS CSI"
    elif provider == "azure":
        driver_label = "app=csi-azuredisk-controller"
        driver_name = "Azure Disk CSI"
    else:
        driver_label = None
        driver_name = "Storage CSI"
    
    if driver_label:
        result = run(
            ["kubectl", "get", "daemonset", "-n", "kube-system", "-l", driver_label, "-o", "json"],
            check=False,
            stream=False
        )
        
        if result.returncode == 0 and result.stdout.strip():
            ds_info = json.loads(result.stdout)
            if ds_info.get("items"):
                ok(f"{driver_name} driver is installed")
                print(f"  DaemonSet: {ds_info['items'][0]['metadata']['name']}")
            else:
                warn(f"{driver_name} driver not found")
                print(f"💡 {driver_name} driver must be installed before deploying LangSmith")
                print("   The Terraform module should handle this, but verify after deployment")
        else:
            warn(f"{driver_name} driver not found")
            print(f"💡 {driver_name} driver must be installed before deploying LangSmith")
    
    # Check StorageClasses
    print("\n### Checking StorageClasses")
    result = run(
        ["kubectl", "get", "storageclass", "-o", "json"],
        check=True,
        stream=False
    )
    sc_list = json.loads(result.stdout)
    
    # Find cloud-specific storage classes
    if provider == "aws":
        storage_scs = [sc for sc in sc_list.get("items", []) if "ebs" in sc["metadata"]["name"].lower() or 
                       sc.get("provisioner", "").endswith("ebs.csi.aws.com")]
    elif provider == "azure":
        storage_scs = [sc for sc in sc_list.get("items", []) if "disk" in sc["metadata"]["name"].lower() or 
                       sc.get("provisioner", "").endswith("disk.csi.azure.com")]
    else:
        storage_scs = []
    
    if storage_scs:
        ok(f"Found {len(storage_scs)} {storage_driver} StorageClass(es):")
        for sc in storage_scs:
            name = sc["metadata"]["name"]
            default = sc.get("metadata", {}).get("annotations", {}).get("storageclass.kubernetes.io/is-default-class", "false")
            print(f"  - {name} (default: {default})")
    else:
        warn(f"No {storage_driver} StorageClasses found")
        print(f"💡 At least one {storage_driver} StorageClass is required for ClickHouse PVCs")
        
except Exception as e:
    warn(f"Could not check storage prerequisites: {e}")
    print("💡 This is expected if the cluster doesn't exist yet")


## Blob Storage Requirement

**Critical:** LangSmith requires cloud object storage (S3, Blob Storage, etc.) for blob storage in production. Inline trace payloads will explode ClickHouse if blob storage is not configured.

Let's verify access to your cloud provider's object storage service and check if a storage account/bucket exists or needs to be created.


In [ ]:
from shared._cloud_helpers import (
    get_cloud_provider,
    get_region,
    get_blob_storage_service_name,
    verify_blob_storage_access,
)
from shared._shell import run
import json

provider = get_cloud_provider()
region = get_region()
blob_service = get_blob_storage_service_name()

print(f"### {blob_service} Access Check")
print(f"Cloud Provider: {provider.upper()}")
print(f"Region: {region}\n")

# Test blob storage access
try:
    if provider == "aws":
        result = run(
            ["aws", "s3", "ls", "--region", region],
            check=True,
            stream=False
        )
        ok(f"{blob_service} access verified")
        
        # List buckets
        buckets_result = run(
            ["aws", "s3api", "list-buckets", "--output", "json"],
            check=True,
            stream=False
        )
        buckets = json.loads(buckets_result.stdout).get("Buckets", [])
        
        print(f"\nFound {len(buckets)} S3 bucket(s):")
        for bucket in buckets[:10]:  # Show first 10
            print(f"  - {bucket['Name']} (created: {bucket['CreationDate']})")
        
        if len(buckets) > 10:
            print(f"  ... and {len(buckets) - 10} more")
        
    elif provider == "azure":
        result = run(
            ["az", "storage", "account", "list", "--output", "json"],
            check=True,
            stream=False
        )
        ok(f"{blob_service} access verified")
        
        # List storage accounts
        accounts = json.loads(result.stdout)
        
        print(f"\nFound {len(accounts)} Storage Account(s):")
        for account in accounts[:10]:  # Show first 10
            name = account.get("name", "N/A")
            location = account.get("location", "N/A")
            print(f"  - {name} (location: {location})")
        
        if len(accounts) > 10:
            print(f"  ... and {len(accounts) - 10} more")
    
    print(f"\n💡 Note: The Terraform module should create a {blob_service} resource for LangSmith blob storage")
    print("   Verify the resource exists after Terraform deployment")
    
except Exception as e:
    warn(f"{blob_service} access check failed: {e}")
    if provider == "aws":
        print("💡 Ensure your AWS credentials have S3 permissions")
    elif provider == "azure":
        print("💡 Ensure your Azure credentials have Storage Account permissions")


## Terraform & Helm Repository Paths

Verify that the Terraform and Helm repository paths are correctly configured and accessible.


In [ ]:
import re
from pathlib import Path
from shared._validation import ok, warn

def expand_env_vars(path_str: str) -> str:
    """Expand environment variable references in a path string."""
    # Expand $VAR and ${VAR} references
    def replace_var(match):
        var_name = match.group(1) or match.group(2)
        return os.environ.get(var_name, match.group(0))
    
    # Replace $VAR and ${VAR} patterns
    path_str = re.sub(r'\$\{([^}]+)\}|\$([a-zA-Z_][a-zA-Z0-9_]*)', replace_var, path_str)
    return path_str

# Expand environment variables in paths (e.g., $TERRAFORM_REPO_DIR, $HELM_REPO_DIR, $HOME)
terraform_dir_str = expand_env_vars(os.environ["TERRAFORM_DIR"])
terraform_dir = Path(terraform_dir_str).expanduser().resolve()

helm_chart_ref_str = expand_env_vars(os.environ["HELM_CHART_REF"])
helm_chart_ref = Path(helm_chart_ref_str).expanduser().resolve()

print("### Repository Paths Check\n")

# Check Terraform directory
print(f"Terraform Directory: {terraform_dir}")
if terraform_dir.exists():
    ok(f"Terraform directory exists")
    
    # Check for main.tf or similar
    tf_files = list(terraform_dir.glob("*.tf"))
    if tf_files:
        print(f"  Found {len(tf_files)} Terraform file(s)")
    else:
        warn("No .tf files found in Terraform directory")
        print("💡 Ensure you're pointing to the correct Terraform module path")
else:
    warn(f"Terraform directory does not exist: {terraform_dir}")
    print("💡 Update TERRAFORM_DIR in your .env file to point to the langchain-ai/terraform repo")

# Check Helm chart
print(f"\nHelm Chart Reference: {helm_chart_ref}")
if helm_chart_ref.exists():
    ok(f"Helm chart path exists")
    
    # Check for Chart.yaml
    chart_yaml = helm_chart_ref / "Chart.yaml"
    if chart_yaml.exists():
        print(f"  Found Chart.yaml")
    else:
        warn("Chart.yaml not found")
        print("💡 Ensure you're pointing to the correct Helm chart path")
else:
    warn(f"Helm chart path does not exist: {helm_chart_ref}")
    print("💡 Update HELM_CHART_REF in your .env file to point to the langchain-ai/helm chart")


## Preflight Summary

Review the checklist below. All items should be ✅ before proceeding to Terraform deployment.

### ✅ Checklist

- [ ] All required tools installed (cloud CLI, terraform, kubectl, helm, jq)
- [ ] Cloud provider credentials valid and correct account/subscription/region
- [ ] Required environment variables set
- [ ] Terraform directory path correct
- [ ] Helm chart path correct
- [ ] Blob storage access verified (S3/Blob Storage)
- [ ] (If cluster exists) Storage CSI driver installed
- [ ] (If cluster exists) StorageClasses configured

### Next Steps

If all checks pass, proceed to **02_terraform_apply.ipynb** to deploy the infrastructure.

If any checks failed, review the warnings above and fix the issues before continuing.
